# 01 — Data collection

    Collect or import the official public source files. Raw files must be saved unchanged in `data/raw/`.

    **Planned sources:** ICNF burned areas, DGT COS/COSc and CAOP, Copernicus DEM GLO-30, and ERA5-Land climate data.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.paths import ensure_output_directories
ensure_output_directories()

print(f"Project root: {PROJECT_ROOT}")

Project root: /mnt/data/wildfire_property_screening_portugal


In [2]:
from src.paths import RAW_DATA_DIR

print(f"Raw data directory: {RAW_DATA_DIR}")
print("No automated downloads are executed. Manually acquired raw archives are registered and validated below without altering them.")

Raw data directory: /mnt/data/wildfire_property_screening_portugal/data/raw
No downloads are executed yet. Source URLs, versions, licences, and collection dates must be logged before data collection.


## Retrospective raw-source registration

The following checks register two official archives acquired manually on 2026-08-03. They read each ZIP in place to calculate its size and SHA-256 checksum and validate its members and CRCs. They do not download, extract, rename, alter, or overwrite raw files.

In [ ]:
from hashlib import sha256
from pprint import pprint
from zipfile import ZipFile


def calculate_sha256(file_path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = sha256()
    with file_path.open("rb") as source_file:
        for chunk in iter(lambda: source_file.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest().upper()


def validate_raw_archive(source: dict) -> dict:
    raw_path = PROJECT_ROOT / source["raw_path"]
    if not raw_path.is_file():
        raise FileNotFoundError(f"Missing raw archive: {raw_path}")
    if raw_path.name != source["filename"]:
        raise ValueError(f"Unexpected filename: {raw_path.name}")

    actual_sha256 = calculate_sha256(raw_path)
    if actual_sha256 != source["expected_sha256"]:
        raise ValueError(f"SHA-256 mismatch for {raw_path.name}: {actual_sha256}")

    with ZipFile(raw_path) as archive:
        archive_members = archive.namelist()
        corrupt_member = archive.testzip()

    missing_members = sorted(set(source["required_members"]) - set(archive_members))
    if corrupt_member is not None:
        raise ValueError(f"ZIP CRC validation failed for member: {corrupt_member}")
    if missing_members:
        raise ValueError(f"Missing required archive members: {missing_members}")

    return {
        "raw_path": str(raw_path.relative_to(PROJECT_ROOT)),
        "filename": raw_path.name,
        "size_bytes": raw_path.stat().st_size,
        "sha256": actual_sha256,
        "zip_integrity": "passed",
        "archive_members": archive_members,
        "required_members_present": True,
    }


def register_and_report(source: dict) -> dict:
    validation = validate_raw_archive(source)
    print(f"Validated immutable raw archive: {source['filename']}")
    print("Provenance:")
    pprint({key: source[key] for key in (
        "dataset_edition_or_year",
        "official_source_url",
        "licence_or_terms_reference",
        "access_date",
        "acquisition_method",
        "raw_path",
        "filename",
        "accepted_facts",
    )}, sort_dicts=False)
    print("Archive validation:")
    pprint(validation, sort_dicts=False)
    return validation

### CAOP 2025 Mainland Portugal boundary

DGT CAOP provides the mainland boundary and municipality reference geography for reporting. The accepted GeoPackage and layer facts below were verified separately using a temporary extraction outside the repository.

In [ ]:
CAOP_2025 = {
    "dataset_edition_or_year": "CAOP 2025 Mainland Portugal boundary",
    "official_source_url": "https://www.dgterritorio.gov.pt/atividades/cartografia/cartografia-tematica/caop",
    "licence_or_terms_reference": "Available without charge (docs/source_plan.md).",
    "access_date": "2026-08-03",
    "acquisition_method": "manual browser download",
    "raw_path": "data/raw/boundaries/dgt_caop/CAOP_Continente_2025-gpkg.zip",
    "filename": "CAOP_Continente_2025-gpkg.zip",
    "expected_sha256": "87CD67F4B1FBADF23D9324E6FB231FF05531E4DB347AF36CCC7C6CBABE3ECD1D",
    "required_members": ["Continente_CAOP2025.gpkg"],
    "accepted_facts": {
        "format": "GeoPackage",
        "crs": "EPSG:3763",
        "cont_nuts1": "one feature where nuts1 == 'Continente'",
        "cont_municipios": "278 features with unique, non-null dtmn",
    },
}

caop_2025_validation = register_and_report(CAOP_2025)

### ICNF 2024 annual burned areas

ICNF annual burned-area cartography supplies historical burned-area geometry. This registration records the verified raw Shapefile facts only; it does not derive features, targets, or a grid.

In [ ]:
ICNF_2024 = {
    "dataset_edition_or_year": "ICNF annual burned areas, 2024",
    "official_source_url": "https://geocatalogo.icnf.pt/catalogo_tema5.html",
    "licence_or_terms_reference": "Public data with attribution requirements (docs/source_plan.md).",
    "access_date": "2026-08-03",
    "acquisition_method": "manual browser download",
    "raw_path": "data/raw/wildfire/icnf_burned_areas/ardida_2024.zip",
    "filename": "ardida_2024.zip",
    "expected_sha256": "B12C74C4D79F928DC55B46977FD0AAF082EDCB05B75474342BB14B4FEC626965",
    "required_members": [
        "ardida_2024.shp",
        "ardida_2024.shx",
        "ardida_2024.dbf",
        "ardida_2024.prj",
    ],
    "accepted_facts": {
        "format": "Shapefile",
        "crs": "EPSG:3763",
        "feature_count": 1558,
        "geometry_quality": "1,558 valid, non-empty features",
        "Ano": "all values equal 2024",
    },
}

icnf_2024_validation = register_and_report(ICNF_2024)